# H03 · Clock trees, PLL math, clock accuracy and timing budgets

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/08_hardware_electronics/H03_clock_trees_pll_math_clock_accuracy_and_timing_b.ipynb)

**Track:** Hardware and electronics · **Skill axis (sheet):** Hardware and electronics · **Time:** about 50 min

**Skill Hub concepts:** Phase-locked loop and clock accuracy; Propagation delay and timing budget

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each low-level task has an optional C/C++ cell after it. Same tests: write the cell, then uncomment the `lab.check(..., "file.c")` line under it. A check passes if either language passes.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("H03")

## Map: where every MHz comes from, and how fast a signal can really go

**STM32G4 main PLL:**
```
 HSE 8 MHz ──/M──> 2.66..16 MHz ──xN──> VCO 96..344 MHz ──/R──> SYSCLK (R in {2, 4, 6, 8})     170 MHz max
 example: 8 MHz / 2 = 4 MHz, x 85 = 340 MHz, / 2 = 170 MHz
```
$$f_{VCO} = f_{in}\,\frac{N}{M}, \qquad f_{SYSCLK} = \frac{f_{VCO}}{R}, \qquad 8 \le N \le 127,\ 1 \le M \le 16$$

| Source | Typical accuracy | Fine for |
|---|---|---|
| Internal RC (HSI) | about +-1% over temperature | UART at modest rates, general timing |
| Crystal (HSE) | 10 to 50 ppm | CAN (needs better than about 0.5% total), USB, Ethernet, long-term timekeeping |
| TCXO | 0.5 to 2 ppm | GNSS, precise timestamps |

Drift: $\text{seconds per day} = \text{ppm} \times 10^{-6} \times 86400$.

**Timing budgets:**
- SPI read where the slave launches data on one edge and the master samples on the next half-cycle:
  $\dfrac{T}{2} \ge t_{CO} + 2\,t_{pd} + t_{SU}$, so $f_{max} = \dfrac{1}{2\,(t_{CO} + 2 t_{pd} + t_{SU})}$
  (clock out over the trace, data back over the trace).
- Synchronous logic (FPGA, ASIC): setup slack $= T + t_{skew} - (t_{clk\to q} + t_{logic} + t_{setup})$; hold slack
  $= t_{clk\to q} + t_{logic,min} - t_{hold} - t_{skew}$. Negative slack = timing violation.

## Task 1: find PLL settings

In [ ]:
def pll_config(f_in, f_target):
    """Return (M, N, R) that hit f_target exactly within the STM32G4 limits, or None. Prefer the smallest M."""
    raise NotImplementedError  # TODO

lab.check("pll", pll_config)

In [ ]:
%%writefile h03_pll.c
// C/C++ version of Task 1 (optional, same tests). Declarations: roboprep/c/include/h03.h
// Grade it with lab.check("pll", "h03_pll.c"). For C++, name the file .cpp in both places.
#include "h03.h"

bool pll_config(double f_in, double f_target, int *m, int *n, int *r) {
    (void)f_in; (void)f_target;
    *m = *n = *r = 0;
    return false; /* TODO: f_ref in 2.66..16 MHz, N in 8..127, VCO in 96..344 MHz, R in {2, 4, 6, 8} */
}

In [ ]:
# lab.check("pll", "h03_pll.c")  # uncomment after writing the C/C++ cell above

## Task 2: accuracy and timing budgets

In [ ]:
def drift_s_per_day(ppm):
    raise NotImplementedError  # TODO

def spi_max_clock(t_co, t_pd, t_su):
    raise NotImplementedError  # TODO

def setup_slack(period, t_cq, t_logic, t_su, skew=0.0):
    raise NotImplementedError  # TODO

def hold_slack(t_cq, t_logic_min, t_hold, skew=0.0):
    raise NotImplementedError  # TODO

lab.check("timing", drift_s_per_day, spi_max_clock, setup_slack, hold_slack)

In [ ]:
%%writefile h03_timing.c
// C/C++ version of Task 2 (optional, same tests). Declarations: roboprep/c/include/h03.h
// Grade it with lab.check("timing", "h03_timing.c"). For C++, name the file .cpp in both places.
#include "h03.h"

double drift_s_per_day(double ppm) { (void)ppm; return 0; /* TODO */ }
double spi_max_clock(double t_co, double t_pd, double t_su) { (void)t_co; (void)t_pd; (void)t_su; return 0; /* TODO */ }
double setup_slack(double period, double t_cq, double t_logic, double t_su, double skew) { (void)period; (void)t_cq; (void)t_logic; (void)t_su; (void)skew; return 0; /* TODO */ }
double hold_slack(double t_cq, double t_logic_min, double t_hold, double skew) { (void)t_cq; (void)t_logic_min; (void)t_hold; (void)skew; return 0; /* TODO */ }

In [ ]:
# lab.check("timing", "h03_timing.c")  # uncomment after writing the C/C++ cell above

## Checkpoint: Math
**M1 (Apply).** A 20 ppm crystal. Seconds of drift per day (3 decimals)?

**M2 (Apply).** Slave clock-to-out 15 ns, 1 ns of trace each way, master setup 5 ns. Maximum SPI clock in MHz (1 decimal)?

**M3 (Apply).** 8 MHz input, M = 2, N = 85. VCO frequency in MHz?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Evaluate).** CAN works on the bench but throws errors on a long bus; the board runs from the internal RC
oscillator. Most likely culprit?
- a) CAN needs a faster CPU
- b) the RC oscillator's +-1% error plus the other nodes' error exceeds CAN's oscillator tolerance, made worse by a long bus; switch to a crystal
- c) too much termination

**K2 (Analyze).** The AS5047P reads fine at 1 MHz but garbage at 10 MHz over a 30 cm ribbon cable. Explain with the
timing budget and list fixes.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `h03.h`).
3. In `roboprep/checks/h03.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()